# Composite Rule Overlap Scoring

This notebook shows the scoring in the simplest possible way:

1. Extracted composite rules are the rules we produced in `comp_output/n1`.
2. Reference composite rules are re-extracted from `PaRoutes/data/n1-routes.json` with the same SynPlanner rule extractor.
3. A composite rule counts as an overlap only when the ordered `$`-joined sequence is present in both sets.
4. Classified alchemical rules from `reference/` tell whether an overlapping composite rule is linked to a positive or negative alchemical rule.

`positive` means the alchemical rule is not equivalent to a default one-step rule. `negative` means its QueryCGR matches a default one-step rule.


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Markdown, SVG, display

pd.set_option("display.max_colwidth", 140)

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "PaRoutes" / "data" / "n1-routes.json").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find Retro-BLEU project root")

COMPOSITE_ROOT = PROJECT_ROOT / "composite_rules"
REFERENCE_DIR = COMPOSITE_ROOT / "reference"

if str(COMPOSITE_ROOT) not in sys.path:
    sys.path.insert(0, str(COMPOSITE_ROOT))

from route_analysis.cli import score_composite_overlap
from synplan.utils.visualisation import get_route_svg_from_json


## Run Scoring

Equivalent terminal command:

```bash
PYTHONPATH=composite_rules \
conda run -n synplan python -m route_analysis.cli score-composite-overlap \
  --extracted-tsv composite_rules/comp_output/n1 \
  --reference-routes-json PaRoutes/data/n1-routes.json \
  --classification-tsv composite_rules/reference/n1_classified_alchemical_rules_pos.tsv composite_rules/reference/n1_classified_alchemical_rules_neg.tsv \
  --output /private/tmp/composite_rule_scores \
  --config composite_rules/configs/rule_extraction_functional_groups.yaml \
  --ignore-errors
```

The `--classification-tsv` files are optional for basic overlap, but they are needed for `pos_overlap` and `neg_overlap`.


In [2]:
EXTRACTED_TSV = COMPOSITE_ROOT / "comp_output" / "n1"
REFERENCE_ROUTES_JSON = PROJECT_ROOT / "PaRoutes" / "data" / "n1-routes.json"
CLASSIFICATION_TSVS = [
    REFERENCE_DIR / "n1_classified_alchemical_rules_pos.tsv",
    REFERENCE_DIR / "n1_classified_alchemical_rules_neg.tsv",
]
OUTPUT_DIR = Path("/private/tmp/composite_rule_scores")
CONFIG = COMPOSITE_ROOT / "configs" / "rule_extraction_functional_groups.yaml"

score_composite_overlap([
    "--extracted-tsv", str(EXTRACTED_TSV),
    "--reference-routes-json", str(REFERENCE_ROUTES_JSON),
    "--classification-tsv", *(str(path) for path in CLASSIFICATION_TSVS),
    "--output", str(OUTPUT_DIR),
    "--config", str(CONFIG),
    "--ignore-errors",
    "--progress-interval", "1000",
])


processed reference routes=1000 reference_composite_rules=3032 errors=0
processed reference routes=2000 reference_composite_rules=4899 errors=0
processed reference routes=3000 reference_composite_rules=6288 errors=0
processed reference routes=4000 reference_composite_rules=7416 errors=0
processed reference routes=5000 reference_composite_rules=8438 errors=0
processed reference routes=6000 reference_composite_rules=9343 errors=0
processed reference routes=7000 reference_composite_rules=10150 errors=0
processed reference routes=8000 reference_composite_rules=10855 errors=0
processed reference routes=9000 reference_composite_rules=11480 errors=0
processed reference routes=10000 reference_composite_rules=12035 errors=0
{
  "extracted_tsvs": [
    "/Users/almazgil/Desktop/projects/Retro-BLEU/composite_rules/comp_output/n1"
  ],
  "reference_routes_json": "/Users/almazgil/Desktop/projects/Retro-BLEU/PaRoutes/data/n1-routes.json",
  "classification_tsvs": [
    "/Users/almazgil/Desktop/projec

0

In [3]:
score_path = OUTPUT_DIR / "composite_rule_overlap_scores.tsv"
matches_path = OUTPUT_DIR / "composite_rule_overlap_matches.tsv"
summary_path = OUTPUT_DIR / "composite_rule_overlap_summary.json"

score = pd.read_csv(score_path, sep="	").iloc[0]
matches = pd.read_csv(matches_path, sep="	")
summary = json.loads(summary_path.read_text())

display(pd.DataFrame([score]))
display(matches.head(10))


,extracted_tsv,reference_routes_json,classification_tsv,extracted_rows,reference_routes,classification_rows,extracted_unique_composite_rules,reference_unique_composite_rules,overlap_unique_composite_rules,classified_overlap_unique_composite_rules,...,extracted_popularity,overlapping_popularity,popularity_overlap_ratio,positive_overlapping_popularity,negative_overlapping_popularity,pos_overlap,neg_overlap,extracted_parse_errors,reference_errors,classification_parse_errors
0,"/Users/almazgil/Desktop/projects/Retro-BLEU/composite_rules/comp_output/n1/n1_t2_composite_rules.tsv,/Users/almazgil/Desktop/projects/Re...",/Users/almazgil/Desktop/projects/Retro-BLEU/PaRoutes/data/n1-routes.json,"/Users/almazgil/Desktop/projects/Retro-BLEU/composite_rules/reference/n1_classified_alchemical_rules_pos.tsv,/Users/almazgil/Desktop/pro...",10208,10000,8765,10208,12035,10203,10143,...,13351,13346,0.999625,13087.126039,193.873961,0.980236,0.014521,0,0,0


,Composite_rule,present_in_reference,classification,classification_positive_weight,classification_negative_weight,classification_positive_share,classification_negative_share,extracted_popularity,extracted_reference_ids,reference_route_ids
0,[C;D2:1]-[N;D2:2]-[C;D3:3](=[O;D1:4])-[c;D3:5]>>[C;D2:1]-[N;D1:2].[C;D3:3](=[O;D1:4])(-[c;D3:5])-[O;D1:6]$[C;D3:1](=[O:2])(-[c:3])-[O;D1...,True,positive,177,0,1.0,0.0,40,"1168,1257,1558,1571,1587,1705,1706,188,1936,1941,2142,2398,2554,2839,3268,3687,3775,4154,438,5927,6,62,6682,6823,6871,6947,7181,7262,747...","188,4154,62,6682,7262,8228,8576"
1,[O;D1:1]=[C;D3:2](-[N;D2:3]-[c;D3:4])-[O;D2:5]>>[N;D1:3]-[c;D3:4].[O;D1:1]=[C;D3:2](-[O;D2:5])-[Cl;D1:6]$[N;D1:1]-[c;D3:2]>>[N;D3+:1](-[...,True,positive,81,0,1.0,0.0,26,"1009,1322,1764,2067,213,2291,2307,2577,2705,2923,3186,4358,5048,5484,578,5959,6010,6495,665,6737,8300,8346,9216,9276,9346,9774",665
2,[O;D1:1]=[C;D3:2](-[N;D2:3]-[c;D3:4])-[c;D3:5]>>[N;D1:3]-[c;D3:4].[O;D1:1]=[C;D3:2](-[c;D3:5])-[Cl;D1:6]$[N;D1:1]-[c;D3:2]>>[N;D3+:1](-[...,True,positive,94,0,1.0,0.0,25,"1384,158,2,2471,3144,3203,3258,3437,3449,3508,3685,4803,5077,523,535,5675,5750,5806,5815,7039,7148,852,8794,8854,932","3203,3685,7148"
3,[C;D1:1]-[S;D4:2](=[O;D1:3])(=[O;D1:4])-[N;D2:5]-[c;D3:6]>>[C;D1:1]-[S;D4:2](=[O;D1:3])(=[O;D1:4])-[Cl;D1:7].[N;D1:5]-[c;D3:6]$[N;D1:1]-...,True,positive,98,0,1.0,0.0,24,"120,146,2021,2443,2463,2752,2878,3289,3684,404,4670,5568,6056,6486,663,6855,689,6929,7323,7566,7654,7665,8489,9273","2021,2878,4670"
4,[O;D1:1]=[S;D4:2](=[O;D1:3])(-[N;D2:4]-[c;D3:5])-[c;D3:6]>>[N;D1:4]-[c;D3:5].[O;D1:1]=[S;D4:2](=[O;D1:3])(-[c;D3:6])-[Cl;D1:7]$[N;D1:1]-...,True,positive,132,0,1.0,0.0,24,"1340,1468,1921,2611,271,2976,2990,3301,3484,3515,3859,4085,4309,4529,4777,4817,5274,66,6716,737,7577,8097,8556,8720",7577
5,[C;D2:1]-[C;D2:2]-[O;D2:3]-[c;D3:4]>>[C;D2:1]-[C;D2:2]-[Br;D1:5].[O;D1:3]-[c;D3:4]$[O;D1:1]-[c;D3:2]>>[O;D2:1](-[c;D3:2])-[C;D1:3],True,positive,79,0,1.0,0.0,19,"1100,1881,2139,2160,3598,4482,5367,575,6022,6208,643,714,718,79,8053,819,911,95,9518","79,8053,819,911"
6,[C;D2:1]-[C;D2:2]-[N;D3:3](-[C;D2:4])-[C;D2:5]>>[N;D2:3](-[C;D2:4])-[C;D2:5].[C;D2:1]-[C;D2:2]-[O;D2:10]-[S;D4:7](-[C:6])(=[O:8])=[O:9]$...,True,positive,77,0,1.0,0.0,18,"1208,1872,2102,2164,2234,4123,431,4778,5534,6250,637,6602,7018,7550,849,8688,9347,9567","1872,2102,637,7018,7550"
7,[C;D2:1]-[C;D3:2](-[N;D3:3](-[C;D2:4])-[C;D2:5])=[O;D1:6]>>[N;D2:3](-[C;D2:4])-[C;D2:5].[C;D2:1]-[C;D3:2](=[O;D1:6])-[O;D1:7]$[C:1]-[C;D...,True,positive,54,0,1.0,0.0,17,"1046,127,250,3266,3473,4282,4298,471,5,51,5461,562,7163,727,8153,9077,919","4282,51"
8,[N;D1:1]-[c;D3:2]>>[N;D3+:1](-[c;D3:2])(=[O;D1:3])-[O;D1-:4]$[N;D3+:1](-[c;D3:2](:[c;D2:3]):[c;D3:4])(=[O;D1:5])-[O;D1-:6]>>[N;D3+:1](=[...,True,positive,47,0,1.0,0.0,17,"1254,1259,1980,2411,261,3176,3579,4357,4386,4897,5274,5442,6457,7014,7566,9473,9997",6457
9,[O;D1:1]=[C;D3:2](-[N;D2:3]-[c;D3:4])-[c;D3:5]>>[N;D1:3]-[c;D3:4].[O;D1:1]=[C;D3:2](-[c;D3:5])-[O;D1:6]$[N;D1:1]-[c;D3:2]>>[N;D3+:1](-[c...,True,positive,75,0,1.0,0.0,17,"1205,1606,1956,2487,2555,2737,2794,4510,508,5117,5912,6138,6518,6971,7536,787,9997",1606


## Score Formula

Let `E` be extracted composite rules and `R` be reference composite rules.

- `overlap = E ∩ R`
- `extracted_overlap_ratio = |overlap| / |E|`
- `reference_coverage_ratio = |overlap| / |R|`
- `jaccard = |overlap| / |E ∪ R|`
- `popularity_overlap_ratio = sum(extracted_popularity(rule) for rule in overlap) / sum(extracted_popularity(rule) for rule in E)`

For alchemical classification, every overlapping composite rule can have positive and/or negative weights from the classified alchemical TSVs.

- `pos_overlap = positive_overlapping_popularity / total_extracted_popularity`
- `neg_overlap = negative_overlapping_popularity / total_extracted_popularity`

So the positive and negative scores are popularity-weighted versions of overlap, not new route extraction procedures.


In [4]:
display(Markdown(
    f"""
For this run:

- `|E| = {int(score.extracted_unique_composite_rules)}`
- `|R| = {int(score.reference_unique_composite_rules)}`
- `|E ∩ R| = {int(score.overlap_unique_composite_rules)}`
- `extracted_overlap_ratio = {score.extracted_overlap_ratio:.6f}`
- `reference_coverage_ratio = {score.reference_coverage_ratio:.6f}`
- `jaccard = {score.jaccard:.6f}`
- `popularity_overlap_ratio = {score.popularity_overlap_ratio:.6f}`
- `pos_overlap = {getattr(score, 'pos_overlap', 0.0):.6f}`
- `neg_overlap = {getattr(score, 'neg_overlap', 0.0):.6f}`
    """
))



For this run:

- `|E| = 10208`
- `|R| = 12035`
- `|E ∩ R| = 10203`
- `extracted_overlap_ratio = 0.999510`
- `reference_coverage_ratio = 0.847777`
- `jaccard = 0.847425`
- `popularity_overlap_ratio = 0.999625`
- `pos_overlap = 0.980236`
- `neg_overlap = 0.014521`
    

## Five Worked Examples

The table below takes five classified alchemical-rule rows from `composite_rules/reference`: three positive and two negative. For each row we choose one linked composite rule and one route ID from `Reference`.

Read each example like this:

- The route contains a composite rule sequence.
- That composite rule is part of an alchemical rule application.
- If the alchemical rule is `positive`, the overlapping composite rule contributes to `pos_overlap`.
- If it is `negative`, it contributes to `neg_overlap`.
- `base_overlap_contribution` is the rule popularity divided by total extracted popularity.


In [5]:
from route_analysis.scoring.overlap import (
    load_composite_rule_classifications,
    load_extracted_composite_rule_set,
    normalize_composite_rule,
    split_composite_rules_cell,
)


def short(text, n=110):
    text = str(text)
    return text if len(text) <= n else text[: n - 3] + "..."


extracted_set = load_extracted_composite_rule_set([EXTRACTED_TSV])
classification_set = load_composite_rule_classifications(CLASSIFICATION_TSVS)
total_extracted_popularity = extracted_set.total_popularity

positive_df = pd.read_csv(CLASSIFICATION_TSVS[0], sep="	")
negative_df = pd.read_csv(CLASSIFICATION_TSVS[1], sep="	")
positive_df["classification"] = "positive"
negative_df["classification"] = "negative"


def first_route_id(row):
    return str(row["Reference"]).split(",")[0]


def first_linked_extracted_composite_rule(row):
    for raw_rule in split_composite_rules_cell(row.get("Composite_rules")):
        try:
            rule = normalize_composite_rule(raw_rule)
        except ValueError:
            continue
        if rule in extracted_set.rules:
            return rule
    return ""


example_rows = []
used_rules = set()
for classification, frame, n_examples in [
    ("positive", positive_df, 3),
    ("negative", negative_df, 2),
]:
    taken = 0
    for _, row in frame.sort_values("popularity", ascending=False).iterrows():
        rule = first_linked_extracted_composite_rule(row)
        if not rule or rule in used_rules:
            continue
        pos_weight, neg_weight = classification_set.weights(rule)
        # For the worked examples, keep negative examples clean when possible:
        # a negative example should mainly show neg_overlap, not a mixed rule.
        if classification == "negative" and pos_weight:
            continue
        used_rules.add(rule)
        route_id = first_route_id(row)
        weight_sum = pos_weight + neg_weight
        extracted_popularity = extracted_set.popularity_by_rule.get(rule, 0)
        positive_share = pos_weight / weight_sum if weight_sum else 0.0
        negative_share = neg_weight / weight_sum if weight_sum else 0.0
        example_rows.append(
            {
                "example": len(example_rows) + 1,
                "classification": classification,
                "route_id": route_id,
                "alchemical_popularity": int(row["popularity"]),
                "composite_extracted_popularity": extracted_popularity,
                "classification_positive_weight": pos_weight,
                "classification_negative_weight": neg_weight,
                "base_overlap_contribution": extracted_popularity / total_extracted_popularity,
                "pos_overlap_contribution": extracted_popularity * positive_share / total_extracted_popularity,
                "neg_overlap_contribution": extracted_popularity * negative_share / total_extracted_popularity,
                "alchemical_rule": row["Alchemical_rule"],
                "composite_rule": rule,
                "target_molecule": str(row["Target_molecules"]).split(",")[0],
            }
        )
        taken += 1
        if taken == n_examples:
            break

examples = pd.DataFrame(example_rows)
display(
    examples[
        [
            "example",
            "classification",
            "route_id",
            "alchemical_popularity",
            "composite_extracted_popularity",
            "classification_positive_weight",
            "classification_negative_weight",
            "base_overlap_contribution",
            "pos_overlap_contribution",
            "neg_overlap_contribution",
        ]
    ]
)


,example,classification,route_id,alchemical_popularity,composite_extracted_popularity,classification_positive_weight,classification_negative_weight,base_overlap_contribution,pos_overlap_contribution,neg_overlap_contribution
0,1,positive,16,59,1,59,0,0.000075,0.000075,0.000000
1,2,positive,6,57,6,57,0,0.000449,0.000449,0.000000
2,3,positive,9,48,1,48,0,0.000075,0.000075,0.000000
3,4,negative,4,27,11,0,27,0.000824,0.000000,0.000824
4,5,negative,162,15,1,0,15,0.000075,0.000000,0.000075


In [12]:
from chython import smarts

routes = json.loads(REFERENCE_ROUTES_JSON.read_text())
routes_json = {idx: route for idx, route in enumerate(routes)} if isinstance(routes, list) else routes

for _, row in examples.iterrows():
    route_id = int(row["route_id"])
    display(Markdown(f"### Example {int(row['example'])}: route `{route_id}` - `{row['classification']}` alchemical rule"))
    display(
        pd.DataFrame(
            [
                {
                    "route_id": route_id,
                    "classification": row["classification"],
                    "how_it_affects_score": (
                        "adds to pos_overlap" if row["classification"] == "positive" else "adds to neg_overlap"
                    ),
                    "base_overlap_contribution": row["base_overlap_contribution"],
                    "pos_overlap_contribution": row["pos_overlap_contribution"],
                    "neg_overlap_contribution": row["neg_overlap_contribution"],
                    "alchemical_rule": short(row["alchemical_rule"], 160),
                    "selected_composite_rule": short(row["composite_rule"], 180),
                }
            ]
        )
    )
    display(SVG(get_route_svg_from_json(routes_json, route_id, labeled=True)))
    display(smarts(row["alchemical_rule"]))


### Example 1: route `16` - `positive` alchemical rule

,route_id,classification,how_it_affects_score,base_overlap_contribution,pos_overlap_contribution,neg_overlap_contribution,alchemical_rule,selected_composite_rule
0,16,positive,adds to pos_overlap,0.000075,0.000075,0.0,[C;D2:1]-[N;D3:2](-[C;D2:3])-[C;D2:4]-[C;D2:5]>>[C;D2:1]-[N;D2:2]-[C;D2:3].[C;D2:4](-[C;D2:5])-[O;D1:6],[C:1]-[C:2](=[C;D2:3]-[C;D2:4]-[N;D3:5](-[C;D2:6])-[C;D2:7])-[C:8]>>[C:1]-[C:2](=[C;D2:3]-[C;D2:4]-[Br;D1:9])-[C:8].[N;D2:5](-[C;D2:6])-[C;D2:7]$[C:1]-[C:2](=[C;D2:3]-[C;D2:4]-...


### Example 2: route `6` - `positive` alchemical rule

,route_id,classification,how_it_affects_score,base_overlap_contribution,pos_overlap_contribution,neg_overlap_contribution,alchemical_rule,selected_composite_rule
0,6,positive,adds to pos_overlap,0.000449,0.000449,0.0,[c;D3:1]-[C;D3:2](=[O;D1:3])-[N;D2:4]-[C;D2:5]>>[c;D3:1]-[C;D3:2](=[O;D1:3])-[O;D2:6]-[C;D1:7].[N;D1:4]-[C;D2:5],[C;D2:1]-[N;D2:2]-[C;D3:3](=[O;D1:4])-[c;D3:5]:[c:6]:[n:7]>>[C;D2:1]-[N;D1:2].[C;D3:3](=[O;D1:4])(-[c;D3:5]:[c:6]:[n:7])-[O;D1:8]$[C;D3:1](=[O:2])(-[c:3])-[O;D1:4]>>[C;D3:1](=[...


### Example 3: route `9` - `positive` alchemical rule

,route_id,classification,how_it_affects_score,base_overlap_contribution,pos_overlap_contribution,neg_overlap_contribution,alchemical_rule,selected_composite_rule
0,9,positive,adds to pos_overlap,0.000075,0.000075,0.0,[c;D3:1]-[C;D3:2](=[O;D1:3])-[N;D2:4]-[c;D3:5]>>[c;D3:1]-[C;D3:2](=[O;D1:3])-[O;D1:6].[N;D3+:4](-[c;D3:5])(=[O;D1:7])-[O;D1-:8],[N;D2:1](-[C;D3:2](=[O;D1:3])-[c;D3:4](:[c:5]:[o:6]):[n:7])-[c;D3:8]>>[C;D3:2](=[O;D1:3])(-[c;D3:4](:[c:5]:[o:6]):[n:7])-[O;D1:9].[N;D1:1]-[c;D3:8]$[N;D1:1]-[c;D3:2]>>[N;D3+:1]...


### Example 4: route `4` - `negative` alchemical rule

,route_id,classification,how_it_affects_score,base_overlap_contribution,pos_overlap_contribution,neg_overlap_contribution,alchemical_rule,selected_composite_rule
0,4,negative,adds to neg_overlap,0.000824,0.0,0.000824,[C;D3:1]-[C;D2:2]-[N;D2:3]=[N;D2+:4]>>[C;D3:1]-[C;D2:2]-[O;D1:5].[N;D1-:3]=[N;D2+:4],[C;D3:1]-[C;D2:2]-[N;D2:3]=[N;D2+:4]>>[C;D3:1]-[C;D2:2]-[O;D2:13]-[S;D4:10](-[c:9]:1:[c:8]:[c:7]:[c:6](-[C:5]):[c:15]:[c:14]:1)(=[O:11])=[O:12].[N;D1-:3]=[N;D2+:4]$[C;D2:1]-[O;...


### Example 5: route `162` - `negative` alchemical rule

,route_id,classification,how_it_affects_score,base_overlap_contribution,pos_overlap_contribution,neg_overlap_contribution,alchemical_rule,selected_composite_rule
0,162,negative,adds to neg_overlap,0.000075,0.0,0.000075,[C;D2:1]-[C;D3:2](=[O;D1:3])-[N;D2:4]-[C;D2:5]>>[N;D1:4]-[C;D2:5].[C;D2:1]-[C;D3:2](=[O;D1:3])-[O;D2:6]-[C;D2:7]-[C:8],[C;D2:1]-[C;D3:2](=[O;D1:3])-[N;D2:4]-[C;D2:5]-[C:6](-[N:7])=[O:8]>>[C;D2:1]-[C;D3:2](=[O;D1:3])-[O;D1:9].[N;D1:4]-[C;D2:5]-[C:6](-[N:7])=[O:8]$[C:1]-[C;D3:2](=[O:3])-[O;D1:4]>...


## One Overlapping Reference Route

In [7]:
overlap_hits = matches[matches['present_in_reference'].astype(str) == 'True'].copy()
hit = overlap_hits.iloc[0]
composite_rule = hit['Composite_rule']
route_id = int(str(hit['reference_route_ids']).split(',')[0])

display(Markdown(f"Selected overlapping composite rule from reference route `{route_id}`."))
display(pd.DataFrame([hit]))


Selected overlapping composite rule from reference route `188`.

,Composite_rule,present_in_reference,classification,classification_positive_weight,classification_negative_weight,classification_positive_share,classification_negative_share,extracted_popularity,extracted_reference_ids,reference_route_ids
0,[C;D2:1]-[N;D2:2]-[C;D3:3](=[O;D1:4])-[c;D3:5]>>[C;D2:1]-[N;D1:2].[C;D3:3](=[O;D1:4])(-[c;D3:5])-[O;D1:6]$[C;D3:1](=[O:2])(-[c:3])-[O;D1...,True,positive,177,0,1.0,0.0,40,"1168,1257,1558,1571,1587,1705,1706,188,1936,1941,2142,2398,2554,2839,3268,3687,3775,4154,438,5927,6,62,6682,6823,6871,6947,7181,7262,747...","188,4154,62,6682,7262,8228,8576"


In [8]:
routes = json.loads(REFERENCE_ROUTES_JSON.read_text())
routes_json = {idx: route for idx, route in enumerate(routes)} if isinstance(routes, list) else routes

def get_route_svg_json(route, route_id=0, labeled=False):
    return get_route_svg_from_json({route_id: route}, route_id, labeled=labeled)

display(SVG(get_route_svg_from_json(routes_json, route_id)))


## Reactions Behind The Overlap

In [9]:
from argparse import Namespace

from route_analysis.composite_rules.extract import (
    ReactionRuleStep,
    RouteProcessingStats,
    SynPlannerRuleExtractor,
    collect_reaction_contexts,
    normalize_route_tree,
    reaction_paths_from_node,
    root_reaction_nodes,
)

extract_args = Namespace(
    config=CONFIG,
    environment_atom_count=1,
    include_rings=False,
    keep_leaving_groups=True,
    keep_incoming_groups=False,
    reactor_validation=False,
)
extractor = SynPlannerRuleExtractor.from_args(extract_args)
route = normalize_route_tree(routes_json[route_id])
step_by_reaction_smiles = {}
stats = RouteProcessingStats()

for reaction_smiles, target_smiles in collect_reaction_contexts(route):
    step, _cache_hit = extractor.extract(reaction_smiles)
    if step is None:
        continue
    step_by_reaction_smiles[reaction_smiles] = ReactionRuleStep(
        rule_smarts=step.rule_smarts,
        center_atoms=step.center_atoms,
        reaction_smiles=step.reaction_smiles,
        target_smiles=target_smiles,
    )

wanted = tuple(composite_rule.split('$'))
matching_segment = None
for root in root_reaction_nodes(route):
    for path in reaction_paths_from_node(root, step_by_reaction_smiles):
        for start in range(len(path)):
            end = start + len(wanted)
            segment = path[start:end]
            if tuple(step.rule_smarts for step in segment) == wanted:
                matching_segment = segment
                break
        if matching_segment:
            break
    if matching_segment:
        break

if matching_segment is None:
    raise RuntimeError('Could not map the selected composite rule back to route reactions.')

pd.set_option('display.max_colwidth', 180)
display(pd.DataFrame([
    {
        'step': index + 1,
        'target_molecule': step.target_smiles,
        'reaction_smiles': step.reaction_smiles,
        'rule_smarts': step.rule_smarts,
        'center_atoms': ','.join(map(str, sorted(step.center_atoms))),
    }
    for index, step in enumerate(matching_segment)
]))


,step,target_molecule,reaction_smiles,rule_smarts,center_atoms
0,1,[cH:13]1[cH:12][c:11]([cH:21][cH:20][c:14]1[CH:15]([CH3:16])[CH2:17][CH2:18][OH:19])[C:9]([NH:8][CH2:7][c:6]2[c:22]([n:24][c:2]([CH3:1])[cH:3][c:4]2[CH3:5])[OH:23])=[O:10],[c:2]1([n:24][c:22]([c:6]([c:4]([CH3:5])[cH:3]1)[CH2:7][NH2:8])[OH:23])[CH3:1].[cH:13]1[cH:12][c:11]([cH:21][cH:20][c:14]1[CH:15]([CH3:16])[CH2:17][CH2:18][OH:19])[C:9](=[O:10]...,[C;D2:1]-[N;D2:2]-[C;D3:3](=[O;D1:4])-[c;D3:5]>>[C;D2:1]-[N;D1:2].[C;D3:3](=[O;D1:4])(-[c;D3:5])-[O;D1:6],"8,9,25"
1,2,[cH:13]1[cH:12][c:11]([cH:21][cH:20][c:14]1[CH:15]([CH3:16])[CH2:17][CH2:18][OH:19])[C:9](=[O:10])[OH:25],[cH:12]1[cH:13][c:14]([cH:20][cH:21][c:11]1[C:9](=[O:10])[O:25][CH3:26])[CH:15]([CH3:16])[CH2:17][CH2:18][OH:19]>>[cH:13]1[cH:12][c:11]([cH:21][cH:20][c:14]1[CH:15]([CH3:16])[C...,[C;D3:1](=[O:2])(-[c:3])-[O;D1:4]>>[C;D3:1](=[O:2])(-[c:3])-[O;D2:4]-[C;D1:5],"25,26"
